<a href="https://colab.research.google.com/github/najahsadmim/llm-diagnostic-syncophancy/blob/main/LLM_Syncophancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Evidence-Grounded Mitigation of Diagnostic Sycophancy in Large Language Models through Confidence Calibration and Threshold-Based Decision Making**

## *Independent Research by NAJAH SADMIM*


##**Import Libraries**

In [1]:
#install libraries
%pip -q install -U openai anthropic pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.5 MB/s eta 0:00:00


In [2]:
import openai
import anthropic
import pydantic

print("OpenAI:", openai.__version__)
print("Anthropic:", anthropic.__version__)
print("Pydantic:", pydantic.__version__)
print("Core imports successful.")

OpenAI: 3.27.0
Anthropic: 1.12.1
Pydantic: 2.14.0
Core imports successful.


In [5]:
#import libraries
import os
import re
import json
import random
import zipfile
import warnings

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
    )

# from sklearn.calibration import (
#     CalibratedClassiferCV,
#     calibration_curve
# )

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from pydantic import BaseModel, Field, ValidationError

from openai import OpenAI
from anthropic import Anthropic
from google import genai

warnings.filterwarnings("default")

SEED=42

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##**Load Dataset**

###**Import Data**

In [6]:
#authorize google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#define paths

DATA_DIR= Path("/content/drive/MyDrive/DDXData")
DATASET_PATH= DATA_DIR / "DDXPlus_30K_Stratified_Research_Dataset.xlsx"
CONDITION_PATH= DATA_DIR / "release_conditions.json"
EVIDENCE_PATH= DATA_DIR / "release_evidences.json"

required_files={
    "30K Dataset": DATASET_PATH,
    "Conditions Metadata": CONDITION_PATH,
    "Evidence Metadata": EVIDENCE_PATH
}

for name, path in required_files.items():
  print(f"{name}: {'Found' if path.exists() else 'NOT FOUND'}")
  print(f"Path: {path}")
  print("-" * 60)

30K Dataset: Found
Path: /content/drive/MyDrive/DDXData/DDXPlus_30K_Stratified_Research_Dataset.xlsx
------------------------------------------------------------
Conditions Metadata: Found
Path: /content/drive/MyDrive/DDXData/release_conditions.json
------------------------------------------------------------
Evidence Metadata: Found
Path: /content/drive/MyDrive/DDXData/release_evidences.json
------------------------------------------------------------


In [8]:
#load 30k dataset

df=pd.read_excel(DATASET_PATH)

print("Dataset loaded successfully)")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully)
Rows: 30,000
Columns: 20


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
#load conditions metadata

with open(CONDITION_PATH, "r", encoding="utf-8") as file:
  conditions_data=json.load(file)

print("Condition metadata loaded successfully.")
print("Metadata type:", type(conditions_data).__name__)
print("Number of top-level entries:", len(conditions_data))

Condition metadata loaded successfully.
Metadata type: dict
Number of top-level entries: 49


In [10]:
#load evidence metadata

with open(EVIDENCE_PATH, "r", encoding="utf-8") as file:
  evidences_data=json.load(file)

print("Evidence metadata loaded successfully.")
print("Metadata type:", type(evidences_data).__name__)
print("Number of evidence entries:", len(evidences_data))

Evidence metadata loaded successfully.
Metadata type: dict
Number of evidence entries: 223


###**Inspect Data**

In [13]:
#dataset inspection

print("Dataset dimensions:", df.shape)

Dataset dimensions: (30000, 20)


In [14]:
print("\nColumn names:")
print(df.columns.tolist())


Column names:
['Case_ID', 'Original_split', 'Source_row_index', 'PATHOLOGY', 'AGE', 'SEX', 'EVIDENCES', 'INITIAL_EVIDENCE', 'DIFFERENTIAL_DIAGNOSIS', 'Evidence_count', 'Num_alternative_diagnoses', 'Has_plausible_differential', 'Differential_alternatives', 'Condition_category', 'DDXPlus_severity_level', 'Urgency_flag', 'ICD10_ID', 'Age_band', 'Evidence_count_band', 'Experimental_subset']


In [15]:
print("\nFirst five rows:")
print(df.head())


First five rows:
           Case_ID Original_split  Source_row_index                            PATHOLOGY  AGE SEX  \
0  DDX_TRN_0005368          train              5368  Acute COPD exacerbation / infection   74   M   
1  DDX_TRN_0005487          train              5487  Acute COPD exacerbation / infection   43   M   
2  DDX_TRN_0005825          train              5825  Acute COPD exacerbation / infection   54   M   
3  DDX_TRN_0007398          train              7398  Acute COPD exacerbation / infection   54   F   
4  DDX_TRN_0013995          train             13995  Acute COPD exacerbation / infection   70   M   

                                                                                                        EVIDENCES  \
0           ['E_31', 'E_66', 'E_72', 'E_77', 'E_79', 'E_123', 'E_125', 'E_198', 'E_201', 'E_204_@_V_10', 'E_214']   
1                           ['E_31', 'E_72', 'E_79', 'E_123', 'E_198', 'E_200', 'E_201', 'E_204_@_V_10', 'E_214']   
2  ['E_31', 'E_66', 'E_7

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
print("\nData types:")
print(df.dtypes.to_frame(name="Data Type"))


Data types:
                           Data Type
Case_ID                       object
Original_split                object
Source_row_index               int64
PATHOLOGY                     object
AGE                            int64
SEX                           object
EVIDENCES                     object
INITIAL_EVIDENCE              object
DIFFERENTIAL_DIAGNOSIS        object
Evidence_count                 int64
Num_alternative_diagnoses      int64
Has_plausible_differential    object
Differential_alternatives     object
Condition_category            object
DDXPlus_severity_level         int64
Urgency_flag                  object
ICD10_ID                      object
Age_band                      object
Evidence_count_band           object
Experimental_subset           object


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
print("\nRandom sample of five cases:")
print(df.sample(5, random_state=42))


Random sample of five cases:
               Case_ID Original_split  Source_row_index                 PATHOLOGY  AGE SEX  \
2308   DDX_TRN_0623567          train            623567        Acute otitis media   62   F   
22404  DDX_TRN_0569032          train            569032        Pulmonary embolism  106   F   
23397  DDX_TRN_0086993          train             86993                       SLE   24   M   
25058  DDX_TST_0044407           test             44407  Scombroid food poisoning   23   F   
2664   DDX_TST_0079903           test             79903        Acute otitis media    9   F   

                                                                                                                     EVIDENCES  \
2308   ['E_53', 'E_54_@_V_161', 'E_54_@_V_192', 'E_55_@_V_130', 'E_56_@_3', 'E_57_@_V_123', 'E_58_@_6', 'E_59_@_6', 'E_91',...   
22404  ['E_34', 'E_45', 'E_53', 'E_54_@_V_192', 'E_55_@_V_55', 'E_55_@_V_127', 'E_55_@_V_128', 'E_55_@_V_159', 'E_55_@_V_17...   
23397  ['E_3', 

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##**Data Preprocessing**

##**Feature Selection**

##**Feature Engineering**

##**Train, Test and Split**

##**Self-Diagnosis Experiment Construction**